# 05-O3 — Build O3 master dataset & candidate feature pool (Bogotá)

**New target: O3**, in parallel to the PM2.5 analysis, toward a combined paper. Reuses
notebooks 01-04's outputs directly (station data, ERA5, fire hotspots, LTS are all
pollutant-agnostic or already include O3/PM10/PM2.5/NO2 columns) -- no need to redo
loading/QC/external-data work.

**Candidates**: PM2.5, PMC (=PM10-PM2.5, the coarse fraction -- a genuinely independent
physical quantity, not a leakage risk the way PM10 was for predicting PM2.5), NO2 and NO
(primary NOx/traffic proxies and direct O3 chemistry participants), plus the same
meteorology/stagnation/fire-hotspot feature set built for PM2.5. **None of these are
excluded here** -- unlike the PM2.5 analysis, there's no definitional-subset relationship
between O3 and any of these; NOx chemistry (NO2 as precursor, titration by NO) and
aerosol-radiation feedback (PM2.5 attenuating the UV/visible flux that drives
photolysis) are textbook, well-established causal mechanisms for O3, not leakage.

**NO is included (v2)**: the local SISAIRE export (`Data/COLOMBIA/BG_*.csv`) has no NO
variable at all, only NO2 -- confirmed by an exhaustive search across all of
`Data/COLOMBIA`. This mattered a great deal because, per the user's own paper (Casallas
et al. 2023, *Air Quality, Atmosphere & Health*, "Surface, satellite ozone variations in
Northern South America during low anthropogenic emission conditions: a machine learning
approach"), **the NO titration effect (NO + O3 -> NO2 + O2) is the single dominant
driver of O3 variability** in Bogotá -- their ML precursor-decomposition attributes
roughly 50-70% of the 2020 ozone changes to the drop in NO, far ahead of NO2, PM2.5,
radiation or wind. NO is pulled from `bogota_no_daily.parquet` (built in notebook 04b
from the same raw hourly station data used in that paper,
`~/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL/NO.csv`, QC'd with the same
Chauvenet/range-check/70%-completeness conventions as the rest of the pipeline). That
source only covers 2013-2022, so NO (and anything derived from it) is NaN for 2023-2026 --
same pattern as the existing fire-hotspot coverage gap.

**Target redefined as MDA8 (v3)**: the user's paper uses the daily maximum 8-hour
running mean (MDA8), the standard WHO/EPA ozone metric (Sokhi et al. 2021 convention),
not a plain daily mean. Checked directly: SISAIRE's published daily `o3` (used as the
target through v2) matches a plain arithmetic daily mean almost exactly (median ratio
~1.00-1.02 vs. a true daily mean computed from hourly data, across 4 stations) and is
consistently 20-65% *lower* than the true MDA8 -- SISAIRE was never reporting MDA8.
`04c_o3_mda8_network.ipynb` computes a proper MDA8 from the same hourly O3 source used
for NO (QC'd per station: range check, Chauvenet on hourly values; 8h rolling mean
requiring >=6/8 valid hours; daily max of that rolling mean, kept only if >=18/24
rolling windows are valid that day; >=70% station-year completeness; network mean of
qualifying stations) -- saved to `bogota_o3_mda8_daily.parquet`. The old SISAIRE plain
daily-mean `o3` is dropped from this notebook entirely (replaced by `o3_mda8` renamed to
`o3`) rather than kept as a separate predictor -- since it's derived from the exact same
underlying hourly series as the new target, keeping it around would be a near-tautological
predictor, the same kind of leakage risk as PM10 was for PM2.5.

**Lesson applied from the PM2.5 work**: greedy sequential feature selection favors a
single clean predictor (raw day-of-year) over many individually-weaker-but-collectively-
-strong ones (fire regions), even when the latter matter more physically. Notebook 06-O3
applies this lesson from the start (a calendar-excluded run) rather than re-discovering
it.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from common.external_data import FIRE_SOURCE_REGIONS, bearing_from_bogota

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
FIG_DIR = PROJECT_ROOT / "Scripts" / "colombia" / "bogota" / "figures"

network = pd.read_parquet(DATA_DIR / "bogota_network_daily_clean.parquet")
era5 = pd.read_parquet(DATA_DIR / "bogota_era5_daily.parquet")
hotspots = pd.read_parquet(DATA_DIR / "bogota_hotspots_daily.parquet")
lts = pd.read_parquet(DATA_DIR / "bogota_lts_daily.parquet")
no = pd.read_parquet(DATA_DIR / "bogota_no_daily.parquet")
co = pd.read_parquet(DATA_DIR / "bogota_co_daily.parquet")  # CO -- combustion proxy, O3-precursor via HO2 cycle
so2 = pd.read_parquet(DATA_DIR / "bogota_so2_daily.parquet")  # SO2 -- combustion/industrial proxy
o3_mda8 = pd.read_parquet(DATA_DIR / "bogota_o3_mda8_daily.parquet")

w700_path = DATA_DIR / "bogota_w700_daily.parquet"
w700 = pd.read_parquet(w700_path) if w700_path.exists() else None

# TROPOMI HCHO -- real satellite-derived VOC proxy (2018-05 onward only, so effective
# coverage over the full 2014-2022 record is ~27% -- likely to be dropped by the
# >=70% coverage filter downstream, but included here for transparency/consistency
# with Medellín, where O3 already starts in 2019 and coverage clears the threshold.
hcho = pd.read_csv(PROJECT_ROOT / "Data" / "external" / "tropomi_hcho" / "bogota_hcho_daily.csv", parse_dates=["date"])
hcho = hcho.rename(columns={"hcho_trop_col_mol_m2": "hcho"})[["date", "hcho"]]

print("network:", network.shape, network['date'].min(), network['date'].max())
print("era5:", era5.shape)
print("hotspots:", hotspots.shape)
print("lts:", lts.shape)
print("no:", no.shape, no['date'].min(), no['date'].max(), "non-null:", no['no'].notna().sum())
print("co:", co.shape, "non-null:", co['co'].notna().sum())
print("so2:", so2.shape, "non-null:", so2['so2'].notna().sum())
print("o3_mda8:", o3_mda8.shape, o3_mda8['date'].min(), o3_mda8['date'].max(),
      "non-null:", o3_mda8['o3_mda8'].notna().sum())
print("w700:", "available" if w700 is not None else "not yet available")
print("o3 (plain daily mean, to be replaced by MDA8) non-null:", network['o3'].notna().sum())

print("hcho:", hcho.shape, "non-null:", hcho['hcho'].notna().sum())

network: (4383, 12) 2014-01-01 00:00:00 2025-12-31 00:00:00
era5: (4384, 12)
hotspots: (4017, 12)
lts: (4384, 3)
no: (3652, 2) 2013-01-01 00:00:00 2022-12-31 00:00:00 non-null: 3650
co: (3652, 2) non-null: 3650
so2: (2557, 2) non-null: 1723
o3_mda8: (3652, 2) 2013-01-01 00:00:00 2022-12-31 00:00:00 non-null: 3650
w700: available
o3 (plain daily mean, to be replaced by MDA8) non-null: 3988
hcho: (2802, 2) non-null: 1445


## Merge

In [2]:
df = (network.merge(era5, on='date', how='left')
             .merge(hotspots, on='date', how='left')
             .merge(lts, on='date', how='left')
             .merge(no, on='date', how='left')
             .merge(co, on='date', how='left')
             .merge(so2, on='date', how='left')
             .merge(o3_mda8, on='date', how='left'))
if w700 is not None:
    df = df.merge(w700, on='date', how='left')
df = df.merge(hcho, on='date', how='left')
df = df.sort_values('date').reset_index(drop=True)

# Replace the target with MDA8 -- drop the old SISAIRE plain-daily-mean o3 entirely
# (derived from the same underlying hourly series, so keeping it as a separate predictor
# would be near-tautological, the same leakage risk PM10 posed for PM2.5).
df = df.drop(columns=['o3']).rename(columns={'o3_mda8': 'o3'})
print(df.shape)


(4383, 41)


## Engineered features

Reuses the exact same dispersion/stagnation/fire-transport engineering built for PM2.5
(BLH, wind, LTS, hotspot regions/sectors are pollutant-agnostic physical mechanisms),
plus two new O3-specific chemical terms:

- `pmc = pm10 - pm25`: the coarse particulate fraction, a genuinely independent
  quantity from PM2.5 (not a subset relationship).
- `photochemical_potential = no2 * radiation_mean`: NOx + sunlight is the textbook O3
  formation mechanism (same idea as `secondary_pm_proxy` in the PM2.5 analysis, but
  here it's the primary mechanism, not an analogy).
- `radiation_effective = radiation_mean / (1 + pm25)`: aerosol loading attenuates the
  UV/visible flux that drives photolysis -- a real physical hypothesis for why more
  PM2.5 might suppress O3 formation, distinct from just offering PM2.5 on its own.
- `no_ratio = no2 / no`: the NO2/NO ratio Casallas et al. (2023) identify as the key
  quantity controlling titration strength -- a smaller ratio (relatively more NO than
  NO2) means more O3 is consumed via NO + O3 -> NO2 + O2, so `no_ratio` should correlate
  *positively* with O3.


In [3]:
df['pmc'] = df['pm10'] - df['pm25']

df['wind_dir_sin'] = np.sin(np.deg2rad(df['wind_dir']))
df['wind_dir_cos'] = np.cos(np.deg2rad(df['wind_dir']))
doy = df['date'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
month = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * month / 12)
df['month_cos'] = np.cos(2 * np.pi * month / 12)
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df['ventilation_station'] = df['blh_mean'] * df['wind_speed']

SECTOR_BEARINGS = {name: i * 45.0 for i, name in enumerate(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])}
point_region_cols = [c for c in df.columns if c.startswith('hotspots_') and not c.startswith('hotspots_sector_')]
sector_cols = [c for c in df.columns if c.startswith('hotspots_sector_')]
for region_col in point_region_cols:
    region_name = region_col.replace('hotspots_', '')
    bearing = bearing_from_bogota(FIRE_SOURCE_REGIONS[region_name]['lat'], FIRE_SOURCE_REGIONS[region_name]['lon'])
    alignment = np.cos(np.deg2rad(df['wind_dir'] - bearing)).clip(lower=0)
    df[f'{region_col}_transport'] = df[region_col] * alignment
for sector_col in sector_cols:
    sector_name = sector_col.replace('hotspots_sector_', '')
    bearing = SECTOR_BEARINGS[sector_name]
    alignment = np.cos(np.deg2rad(df['wind_dir'] - bearing)).clip(lower=0)
    df[f'{sector_col}_transport'] = df[sector_col] * alignment

df['ventilation_min'] = df['blh_min'] * df['wind_speed']
df['fog_stagnation'] = df['rh_2m'] / (1 + df['wind_speed'])
df['dry_stagnation'] = 1 / ((1 + df['wind_speed']) * (1 + df['precip_total_mm']))

# O3-specific chemistry terms.
df['photochemical_potential'] = df['no2'] * df['radiation_mean']
df['radiation_effective'] = df['radiation_mean'] / (1 + df['pm25'])
df['no_ratio'] = np.where(df['no'] > 1e-3, df['no2'] / df['no'], np.nan)


In [4]:
LAG_DAYS = [1, 2, 3]
hotspot_cols = [c for c in df.columns if c.startswith('hotspots_')]

LAG_VARS = [
    'pm25', 'pmc', 'no2', 'no', 'no_ratio', 'co', 'so2', 'wind_speed', 'temp_2m', 'rh_2m', 'pressure',
    'blh_mean', 'blh_morning_mean', 'radiation_mean', 'precip_total_mm',
    'ventilation_station', 'lts', 'lts_morning', 'ventilation_min',
    'fog_stagnation', 'dry_stagnation', 'photochemical_potential', 'radiation_effective',
    'tcwv', 'sensible_heat_flux', 'tke_proxy', 'w700', 'hcho',
] + hotspot_cols
df = df.set_index('date')
for var in LAG_VARS:
    if var not in df.columns:
        continue
    for lag in LAG_DAYS:
        df[f'{var}_lag{lag}'] = df[var].shift(lag)

df['o3_lag1'] = df['o3'].shift(1)
df = df.reset_index()

candidate_cols = [c for c in df.columns if c not in ('date', 'o3')]
print(f"{len(candidate_cols)} candidate predictor columns")


217 candidate predictor columns


/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_85470/259508498.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{var}_lag{lag}'] = df[var].shift(lag)
/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_85470/259508498.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{var}_lag{lag}'] = df[var].shift(lag)
/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_85470/259508498.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

## Verification: do the new O3-specific mechanisms behave as expected?

`no2`, `photochemical_potential` should correlate *positively* with O3 (NOx + sunlight
drives formation, up to the NOx-saturation regime); `no` should correlate *negatively*
with O3 (titration consumes O3) and be the single strongest precursor relationship per
Casallas et al. (2023); `no_ratio` should correlate *positively* (less NO relative to
NO2 means weaker titration); `radiation_effective` should correlate more strongly
positively with O3 than raw `radiation_mean` if the aerosol-attenuation hypothesis has
any signal; `pm25`/`pmc` relationships are genuinely uncertain a priori (could go either
way depending on attenuation vs. shared-source confounding) -- reported, not assumed.


In [5]:
for col in ['pm25', 'pmc', 'no2', 'no', 'no_ratio', 'co', 'so2', 'radiation_mean', 'photochemical_potential', 'radiation_effective', 'wind_speed', 'lts']:
    if col in df.columns:
        corr = df[['o3', col]].dropna().corr().iloc[0, 1]
        n = df[['o3', col]].dropna().shape[0]
        print(f"corr(o3, {col}) = {corr:.3f}  (n={n})")


corr(o3, pm25) = 0.487  (n=3285)
corr(o3, pmc) = 0.218  (n=3284)
corr(o3, no2) = 0.373  (n=2920)
corr(o3, no) = -0.247  (n=3285)
corr(o3, no_ratio) = 0.483  (n=2920)
corr(o3, co) = 0.008  (n=3285)
corr(o3, so2) = -0.046  (n=1722)
corr(o3, radiation_mean) = 0.450  (n=3285)
corr(o3, photochemical_potential) = 0.514  (n=2920)
corr(o3, radiation_effective) = -0.183  (n=3285)
corr(o3, wind_speed) = 0.082  (n=2920)
corr(o3, lts) = 0.114  (n=3285)


## Save O3 master dataset

In [6]:
out_path = DATA_DIR / "bogota_o3_master_dataset.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path, df.shape)
print("Rows with non-null target (o3):", df['o3'].notna().sum())


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_o3_master_dataset.parquet (4383, 219)
Rows with non-null target (o3): 3285
